# Synthetic campaign — which strategy, and at what settings

Sweeps the routing policies over the campus map and files every run in the
[results store](../autobus-docs/docs/concepts/experiment.md), so this notebook
can be **stopped at any point without losing work** and re-run later to
continue. Nothing here needs a CARLA server.

The design review's statistical bar — a confidence-interval half-width under
5% of the mean for every KPI — lives *here*, on the synthetic map, rather than
on CARLA. CARLA's job is to show the result carries over; this is where the
result is established.

## How to run it

**Prefer a terminal over this notebook for the long stages.** Workers are
forked, and forking a process that already has threads can deadlock a child.
A script run through `run.sh` is single-threaded; a Jupyter kernel is not.
Everything below works in Jupyter and usually finishes, but for an unattended
sweep, export this to a script.

To stop: interrupt the kernel. Every trial that finished is already on disk.
To continue: re-run the same cell — it reruns only what is missing.

## Two knobs both called `max_wait_s`

They are unrelated and it matters:

| | what it is |
|---|---|
| `run_trial(max_wait_s=...)` | **passenger patience** — how long someone waits before giving up. `None` means they wait forever. |
| `WindowInsertionPolicy(max_wait_s=...)` | the **service window** — the promise the policy plans against, and refuses requests it cannot keep. |

Below they are called `PATIENCE` and `W` respectively.

## Configuration

In [ ]:
from src.experiment.store import DEFAULT_RUNS_DIR

# --- The map every experiment in this project has used ------------------
MAP = dict(n_seeds=17, bounds=(0.0, 0.0, 2000.0, 2000.0), seed=42)

# --- Where runs are filed -----------------------------------------------
# DEFAULT_RUNS_DIR is anchored to the source tree, not to the working
# directory: JupyterLab starts a kernel in the notebook's folder, and a
# relative path would file half a campaign somewhere else entirely.
RUNS = DEFAULT_RUNS_DIR

# --- Sample size ---------------------------------------------------------
# Start here; the last section reports which KPIs missed the 5% bar and
# extending is free, because the store reruns only what is missing.
SEEDS = list(range(60))

# --- Load ----------------------------------------------------------------
# 200 requests at one per 60 s, capacity 40: the settings the existing
# notebooks used, so these runs extend that work rather than replace it.
BASE_TRIAL = dict(
    max_requests=200,
    capacity=40,
    demand_interval_s=60.0,
    max_time=60000.0,
)

# --- The two regimes -----------------------------------------------------
# Measured before choosing: with PATIENCE=None every policy serves 100% and
# fulfilment rate discriminates nothing at all. At 900 s the field spreads
# from 80% to 100%, which is also where the design review's ">= 80%"
# criterion starts to mean something.
REGIMES = {
    "patient": None,
    "impatient": 900.0,
}

# --- Service windows to sweep -------------------------------------------
WINDOWS = [450.0, 600.0, 750.0, 900.0, 1200.0, 1800.0]
DEFAULT_W = 900.0

# --- Parallelism ---------------------------------------------------------
# 8 measured as the sweet spot on a 16-core machine; the work is CPU-bound
# pure Python, so hyperthreads add little. Use 1 to debug.
N_JOBS = 8

# --- Stage 3 is opt-in; it is the longest ---------------------------------
RUN_LOAD_SWEEP = False
LOAD_INTERVALS = [30.0, 45.0, 60.0, 90.0]

In [ ]:
import time

import matplotlib.pyplot as plt

from src.algorithm.ConstrainedInsertionPolicy import ConstrainedInsertionPolicy
from src.algorithm.DeviatedFixedRoutePolicy import DeviatedFixedRoutePolicy
from src.algorithm.FixedRoutePolicy import FixedRoutePolicy
from src.algorithm.MarginalCostFilterPolicy import MarginalCostFilterPolicy
from src.algorithm.NearestUnservedPolicy import NearestUnservedPolicy
from src.algorithm.ReorderingInsertionPolicy import ReorderingInsertionPolicy
from src.algorithm.RideBoundedGreedyPolicy import RideBoundedGreedyPolicy
from src.algorithm.SkipStopFixedRoutePolicy import SkipStopFixedRoutePolicy
from src.algorithm.WindowInsertionPolicy import WindowInsertionPolicy
from src.experiment import (
    plot_kpi_bars,
    plot_parameter_sweep,
    run_campaign,
    summarize_results,
)
from src.experiment.store import load_frame
from src.map.graph import PlanarStreetGraph

from IPython.display import Markdown, display

graph = PlanarStreetGraph.generate(**MAP)

# The part of a run's identity that cannot be read off the arguments. The map
# is described by its generation parameters because PlanarStreetGraph does not
# retain them -- leave one out and two different maps would share an identity.
WORLD = {
    "map": "planar",
    "n_seeds": MAP["n_seeds"],
    "bounds": list(MAP["bounds"]),
    "map_seed": MAP["seed"],
    "backend": "MockBackend",
}

print(f"map: {MAP['n_seeds']} seeds over {MAP['bounds'][2]:.0f} m, "
      f"seed {MAP['seed']}")
print(f"{len(SEEDS)} seeds per configuration, n_jobs={N_JOBS}")

## The policies

All nine concrete policies. `PlaceholderPolicy` is excluded — it is a
development stub, not a strategy.

The two insertion policies are labelled with their service window, so the
window sweep in stage 2 reuses these runs rather than repeating them.

In [ ]:
def window_family(windows):
    """
    Build the insertion policies at each service window.
    :param windows: Service windows in seconds.
    :return: (label, factory) pairs.
    """
    pairs = []
    for w in windows:
        pairs.append((f"Window W{w:.0f}",
                      lambda g, w=w: WindowInsertionPolicy(g, max_wait_s=w)))
        pairs.append((f"Reordering W{w:.0f}",
                      lambda g, w=w: ReorderingInsertionPolicy(
                          g, max_wait_s=w)))
    return pairs


BASELINES = [
    ("Fixed route", lambda g: FixedRoutePolicy(g)),
    ("Skip-stop", lambda g: SkipStopFixedRoutePolicy(g)),
    ("Deviated", lambda g: DeviatedFixedRoutePolicy(g)),
]
GREEDY = [
    ("Nearest unserved", lambda g: NearestUnservedPolicy(g)),
    ("Marginal cost", lambda g: MarginalCostFilterPolicy(g)),
    ("Ride bounded", lambda g: RideBoundedGreedyPolicy(g)),
]
INSERTION = [("Constrained", lambda g: ConstrainedInsertionPolicy(g))]

SHOOTOUT = (BASELINES + GREEDY + INSERTION
            + window_family([DEFAULT_W]))

print(f"{len(SHOOTOUT)} policies in the shootout")
for label, _ in SHOOTOUT:
    print(" ", label)

## Calibration — how long will this take?

Times one trial of each policy so the estimate below is measured rather than
guessed. `ReorderingInsertionPolicy` dominates the cost; the fixed-route
family is nearly free.

In [ ]:
def time_one(factory, patience):
    """
    Time a single trial.
    :param factory: Policy factory.
    :param patience: Passenger patience in seconds, or None.
    :return: Seconds elapsed.
    """
    from src.experiment import run_trial
    started = time.perf_counter()
    run_trial(policy_factory=factory, graph=graph, seed=0,
              max_wait_s=patience, silent=True, **BASE_TRIAL)
    return time.perf_counter() - started


per_trial = {label: time_one(f, REGIMES["impatient"]) for label, f in SHOOTOUT}
for label, seconds in sorted(per_trial.items(), key=lambda kv: -kv[1]):
    print(f"  {label:<20} {seconds:6.2f} s/trial")

stage1 = sum(per_trial.values()) * len(SEEDS) * len(REGIMES)
extra = [f"{k} W{w:.0f}" for w in WINDOWS if w != DEFAULT_W
         for k in ("Window", "Reordering")]
stage2 = (per_trial[f"Window W{DEFAULT_W:.0f}"]
          + per_trial[f"Reordering W{DEFAULT_W:.0f}"]) \
    * (len(WINDOWS) - 1) * len(SEEDS) * len(REGIMES)

print()
print(f"stage 1  ~{stage1 / 60:5.1f} min sequential  "
      f"~{stage1 / 60 / 6.8:5.1f} min at n_jobs={N_JOBS}")
print(f"stage 2  ~{stage2 / 60:5.1f} min sequential  "
      f"~{stage2 / 60 / 6.8:5.1f} min at n_jobs={N_JOBS}")
print("(already-recorded runs are skipped, so a re-run costs nothing)")

## Stage 1 — the shootout

Every policy, both regimes. This is the table that says which *family* wins,
and it is the evidence for setting the others aside.

In [ ]:
def sweep(policies, regimes=REGIMES, **trial_overrides):
    """
    Run one campaign per regime and report what it did.
    :param policies: (label, factory) pairs.
    :param regimes: Mapping of regime name to passenger patience.
    :param trial_overrides: Trial parameters replacing BASE_TRIAL entries.
    :return: Mapping of regime name to CampaignResult.
    """
    trial = {**BASE_TRIAL, **trial_overrides}
    results = {}
    for name, patience in regimes.items():
        started = time.perf_counter()
        results[name] = run_campaign(
            policies=policies,
            seeds=SEEDS,
            world=WORLD,
            directory=RUNS,
            graph=graph,
            n_jobs=N_JOBS,
            progress=True,
            max_wait_s=patience,
            **trial,
        )
        print(f"{name:<10} ran={results[name].ran:<5} "
              f"skipped={results[name].skipped:<5} "
              f"{(time.perf_counter() - started) / 60:.1f} min")
    return results


shootout = sweep(SHOOTOUT)

In [ ]:
for name, result in shootout.items():
    patience = REGIMES[name]
    display(Markdown(
        f"### {name} — passenger patience "
        f"{'unlimited' if patience is None else f'{patience:.0f} s'}"))
    display(Markdown(summarize_results(result.results()).markdown))

### Reading it

The `impatient` table is the one that discriminates. With unlimited patience
every policy eventually serves everyone, so **fulfilment rate is pinned at
100% and says nothing** — which is exactly why both regimes are run.

In [ ]:
fig = plot_kpi_bars(
    shootout["impatient"].results(),
    fields=["served_fraction", "mean_wait_time", "mean_ride_time",
            "throughput"],
    title="Shootout under load (patience 900 s)",
)
plt.show()

## Stage 2 — the service window

Only the insertion family has a window to tune. `W900` already ran in
stage 1, so the store skips it here.

In [ ]:
window_sweep = sweep(window_family(WINDOWS))

In [ ]:
frame = load_frame(RUNS)
frame = frame[
    (frame["identity.world.map_seed"] == MAP["seed"])
    & (frame["identity.trial.max_requests"] == BASE_TRIAL["max_requests"])
    & (frame["identity.trial.demand_interval_s"]
       == BASE_TRIAL["demand_interval_s"])
    & frame["identity.policy_params.max_wait_s"].notna()
].copy()
# One line per policy family rather than per label, so the window becomes the
# x axis instead of nine separate columns.
frame["family"] = frame["identity.policy_class"].str.replace(
    "InsertionPolicy", "", regex=False)

for patience, label in ((None, "patient"), (900.0, "impatient")):
    subset = frame[frame["identity.trial.max_wait_s"].isna()
                   if patience is None
                   else frame["identity.trial.max_wait_s"] == patience]
    if subset.empty:
        continue
    for metric in ("kpi.mean_wait_time", "kpi.mean_ride_time",
                   "kpi.served_fraction"):
        plot_parameter_sweep(
            subset,
            x="identity.policy_params.max_wait_s",
            y=metric,
            hue="family",
            title=f"{metric.split('.')[-1]} vs service window ({label})",
        )
        plt.show()

## Stage 3 — load (optional)

How each leader degrades as demand outpaces the bus. Set `RUN_LOAD_SWEEP` in
the configuration cell to enable; it is the longest stage.

In [ ]:
if RUN_LOAD_SWEEP:
    for interval in LOAD_INTERVALS:
        print(f"--- one request every {interval:.0f} s ---")
        sweep(window_family([DEFAULT_W]) + [BASELINES[0]],
              demand_interval_s=interval)
else:
    print("RUN_LOAD_SWEEP is off")

## Did we meet the statistical bar?

The design review asks for a confidence-interval half-width below **5% of the
mean** on every KPI. This reports what was achieved and how many seeds each
shortfall would need.

Extending is free: raise `SEEDS` in the configuration cell and re-run stage 1.
Everything already recorded is skipped.

In [ ]:
import math
import statistics

from scipy import stats

METRICS = {
    "served fraction": lambda r: r.kpi.served_fraction,
    "mean wait": lambda r: r.kpi.mean_wait_time,
    "mean ride": lambda r: r.kpi.mean_ride_time,
    "throughput": lambda r: r.kpi.throughput,
}


def ci_report(results, confidence=0.95):
    """
    Report the achieved CI half-width per policy and KPI.
    :param results: Policy label to trial results.
    :param confidence: Confidence level.
    :return: Markdown table.
    """
    lines = ["| policy | KPI | mean | CI half | % of mean | seeds for 5% |",
             "| --- | --- | --- | --- | --- | --- |"]
    for label, runs in sorted(results.items()):
        for name, get in METRICS.items():
            values = [get(r) for r in runs if get(r) is not None]
            n = len(values)
            if n < 2:
                continue
            mean = statistics.fmean(values)
            sd = statistics.stdev(values)
            if mean == 0 or sd == 0:
                lines.append(f"| {label} | {name} | {mean:.3g} | 0 | 0.0% | "
                             f"met |")
                continue
            half = stats.t.ppf(1 - (1 - confidence) / 2, n - 1) * sd / n ** 0.5
            pct = 100 * half / abs(mean)
            need = "met" if pct <= 5 else str(math.ceil(n * (pct / 5) ** 2))
            lines.append(f"| {label} | {name} | {mean:.3g} | {half:.3g} | "
                         f"{pct:.1f}% | {need} |")
    return "\n".join(lines)


display(Markdown("### Under load (the regime that discriminates)"))
display(Markdown(ci_report(shootout["impatient"].results())))

## What is now in the store

Everything above, plus anything recorded earlier — including the CARLA runs.
`notebooks/results.ipynb` reads the same store and needs none of this to have
just run.

In [ ]:
inventory = load_frame(RUNS)
print(f"{len(inventory)} runs on disk")
by_world = inventory.groupby(
    ["identity.world.backend", "identity.policy"]
).size().sort_values(ascending=False)
display(by_world.to_frame("runs"))